# Initial-shock variability: E3SM and CESM-SMYLE

Archive-backed companion to `1a_refactor_atm_leadtime_acc_skill_map.ipynb`.

**Configure → inventory and cache plan → monthly archive preparation and common-grid
calculation → cached diagnostics → case comparisons → individual initialization plots.**

This uses the same E3SM post-processing archive, CESM-SMYLE benchmark archive,
observation accessors and regridding utilities as `1a`. The calculation is separate:
no detrending, lead-drift removal, or ACC anomaly inputs. The scalar ratio is the
model ensemble-mean temporal standard deviation divided by observed standard deviation.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'esp_lab').is_dir()), None)
if repo_root is None:
    raise RuntimeError('Run this notebook from the repository root or jupyter directory')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from importlib import reload
from esp_lab.diagnostics import initial_shock as initial_shock_diagnostic
from workflows.diagnostics import initial_shock_archive as initial_shock_archive_workflow
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.resource_utils import ResourceTracker

initial_shock_diagnostic = reload(initial_shock_diagnostic)
plot_std_ratio = initial_shock_diagnostic.plot_std_ratio
plot_normalized_change = initial_shock_diagnostic.plot_normalized_change

if initial_shock_archive_workflow.ARCHIVE_VERSION != 'initial_shock_archive_v2':
    initial_shock_archive_workflow = reload(initial_shock_archive_workflow)
plan_archive_run = initial_shock_archive_workflow.plan_archive_run
compute_archive_plan = initial_shock_archive_workflow.compute_archive_plan


## User setup and workflow configuration

Select a field and the E3SM cases here, as in `1a`. Input files are discovered through
the existing archive accessors; no manually prepared monthly filenames are needed.
`TREFHT` is the temperature diagnostic used in the NCL example. All conversion
factors below assume the same archive conventions as `1a`; PRECT is mm/day after conversion.

The original NCL index needs five annual samples (60 months). With the current
24-month archive, these defaults explicitly compute an **exploratory two-block
ratio**, using two consecutive 12-month means. May blocks are May–April; November
blocks are November–October. This is not the original five-year statistic.


In [2]:

VAR_CONFIG = {
    'TREFHT': dict(obs_product='ERA5', obs_variable='tas', units='degC',
                   model_scale=1., model_offset=-273.15, smyle_scale=1., smyle_offset=-273.15,
                   obs_scale=1., obs_offset=-273.15),
    'TS': dict(obs_product='ERA5', obs_variable='ts', units='degC',
               model_scale=1., model_offset=-273.15, smyle_scale=1., smyle_offset=-273.15,
               obs_scale=1., obs_offset=-273.15),
    'PRECT': dict(obs_product='GPCP_v2.3', obs_variable='PRECT', units='mm/day',
                  model_scale=86400000., model_offset=0., smyle_scale=86400000., smyle_offset=0.,
                  obs_scale=1., obs_offset=0.),
    'PSL': dict(obs_product='ERA5', obs_variable='psl', units='hPa',
                model_scale=.01, model_offset=0., smyle_scale=.01, smyle_offset=0.,
                obs_scale=.01, obs_offset=0.),
}

field = 'PRECT' #'PSL' #'TS' #'TREFHT'
variable = dict(VAR_CONFIG[field], field=field, model_variable=field, smyle_variable=field)

# Optional common processing end year. None uses the full period.
YEAR_END = 2011  # None

clim1=1981
clim2=2010
init_months = [5, 11]
nlead = 24

target_dlat = 5.0
target_dlon = 5.0
regrid_method = 'conservative'
periodic = True

# Explicit color-bin boundaries for the normalized-change heatmaps.
HEATMAP_COLORBAR_LEVELS = {
    'signed_normalized_change': np.linspace(-2.0, 2.0, 11),
    'absolute_normalized_change': np.linspace(0.0, 2.0, 11),
}

E3SM_CASES = {
    'E3SM-FOSIRL': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL', cache_tag='JRA55_FOSIRL'),
    'E3SM-Reanalysis': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce', cache_tag='Reanalysis'),
    'E3SM-4DEnVarOcn': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn', cache_tag='4DEnVarOcn'),
}

WORKFLOW_SETTINGS = {
    'paths': {
        's2d_diag_root': '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag',
        'figure_outdir': '/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/initial_shock',
    },
    'run': {
        'years': [1980, 2018 if YEAR_END is None else int(YEAR_END)],
        'init_months': init_months, 
        'nlead': nlead, 
        'smoke_mode': False
    },
    'e3sm': {
        'data_dir': '/global/cfs/cdirs/e3sm/S2S2D/post_process',
        'nens': 10, 'grid': '180x360_aave', 'ts_split': '2yr', 'engine': 'netcdf4',
        'chunks': {'Y': 3, 'L': 24, 'M': 2, 'lat': 90, 'lon': 180},
    },
    'smyle': {
        'include': True, 'nens': 20,
        'benchmark_dir': '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE',
        'chunks': {'Y': 3, 'L': 24, 'M': 2, 'lat': 96, 'lon': 144},
    },
    'obs': {
        'data_dir': '/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series',
        'chunks': {'time': 24, 'lat': 90, 'lon': 180},
    },
    'regrid': {
        'target_dlat': target_dlat, 
        'target_dlon': target_dlon, 
        'method': regrid_method, 
        'periodic': periodic
    },
    'metric': {
        'window_months': 24, 
        'block_months': 12, 
        'start_lead': 0,
        'min_samples': None, 
        'min_area_fraction': .9,
        'climatology_years': [clim1, clim2]
    },
    'cache': {
        'mode': 'auto', 
        'force_compute': False
    },  # set force_compute=True to refresh valid caches
    'dask': {
        'enabled': True, 
        'cluster_type': 'local',
        'workers': 4,
        'memory_limit': '4GB'
    },
}

if WORKFLOW_SETTINGS['run']['smoke_mode']:
    E3SM_CASES = {'E3SM-FOSIRL': E3SM_CASES['E3SM-FOSIRL']}
    WORKFLOW_SETTINGS['run'].update(years=[1980, 1981], init_months=[11])
    WORKFLOW_SETTINGS['regrid'].update(target_dlat=5., target_dlon=5.)
    WORKFLOW_SETTINGS['dask']['workers'] = 2
    print('Archive-backed smoke mode: one E3SM case, CESM-SMYLE, two starts, one month.')
    
print('Field:', field, '| Years:', WORKFLOW_SETTINGS['run']['years'])
print('Metric:', WORKFLOW_SETTINGS['metric'])


Field: PRECT | Years: [1980, 2011]
Metric: {'window_months': 24, 'block_months': 12, 'start_lead': 0, 'min_samples': None, 'min_area_fraction': 0.9, 'climatology_years': [1981, 2010]}


## Inventory and cache plan

Inventory all requested initializations before starting expensive processing.
All E3SM members are required. Missing initializations cause an error rather than
silently shortening a case's comparison period. This also checks CESM-SMYLE
**monthly** benchmark availability and identifies the observation source.

Identity includes source file inventories, settings and adapter code. Compatible
metric caches can be reused without remapping the data. `require` needs source
metadata and existing compatible caches. Rerun this cell after changing settings.


In [ ]:
archive_plan = plan_archive_run(WORKFLOW_SETTINGS, E3SM_CASES, variable)
plan_table = pd.DataFrame([
    {'case': t['case'], 'init_month': t['month'], 'initializations': len(t['years']),
     'source_files': len(t['inventory']), 'action': 'prepare' if t['rebuild'] else 'reuse',
     'cache': t['path']}
    for t in archive_plan
])
display(plan_table)


## Dask resources

Start after the inventory succeeds. A local cluster starts only when the plan contains
work to rebuild; rerunning this cell closes any previous cluster. The computation cell
always releases Dask resources, including when computation raises an exception.


In [ ]:
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

needs_distributed_compute = (
    WORKFLOW_SETTINGS['dask']['enabled']
    and any(task['rebuild'] for task in archive_plan)
)
cluster, client, workflow_resources = restart_notebook_cluster(
    globals(),
    lambda: get_cluster_client(DaskConfig(
        cluster_type=WORKFLOW_SETTINGS['dask']['cluster_type'],
        workers=WORKFLOW_SETTINGS['dask']['workers'],
        memory_limit=WORKFLOW_SETTINGS['dask']['memory_limit'],
    )) if needs_distributed_compute else (None, None),
)
if client is not None:
    display(client)
elif WORKFLOW_SETTINGS['dask']['enabled']:
    print('All exact-period caches are valid; skipped distributed cluster startup.')


## Monthly preparation, common-grid calculation, and cache writing

For each missing case/month product, the module opens monthly archives, validates
represented verification months, converts units, regrids model and observation,
and computes the metric. Each source dataset is closed after the calculation.

To save space, the preparation and metric stages write **compact global block
indices, coverage and normalized changes**, not duplicate gridded monthly files. These live
under `<case>/initial_shock/metrics/atm/<field>/`, separate from ACC/RMSE outputs.
The normalized-change denominator uses unique observed first-block annual means
over `metric.climatology_years`: May-April for May starts and November-October
for November starts.


In [ ]:
try:
    comparison_by_month = compute_archive_plan(archive_plan, WORKFLOW_SETTINGS, variable)
finally:
    close_notebook_resources(globals())
    print('Closed Dask resources after computation.')
for month, comparison in comparison_by_month.items():
    print(f'Initialization month {month:02d}')
    display(comparison[['signed_normalized_change', 'absolute_normalized_change',
                        'excess_normalized_change', 'observation_climatology_std',
                        'std_ratio', 'paired_sample_count', 'valid_normalized_change']])


## Coverage, normalized-change tables and heatmaps

The primary signed metric is `(M2-M1) / sigma_obs_clim`; positive values indicate
an increase from lead year 1 to lead year 2 and negative values indicate a decrease.
The secondary absolute metric shows adjustment magnitude. May and November share
a figure and color scale for each metric. The legacy two-point std ratio remains in
the tables for continuity but is not the primary diagnostic.


In [ ]:
FIGURE_ROOT = Path(WORKFLOW_SETTINGS['paths']['figure_outdir'])
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

period_start, period_end = WORKFLOW_SETTINGS['run']['years']
period_token = f'{period_start}_{period_end}'

for month, comparison in comparison_by_month.items():
    prefix = f'{field}_init{month:02d}_{period_token}'
    summary = comparison[['signed_normalized_change', 'absolute_normalized_change',
                          'excess_normalized_change', 'model_lead_year_change',
                          'observation_lead_year_change', 'observation_climatology_std',
                          'observation_climatology_sample_count', 'std_ratio',
                          'paired_sample_count', 'valid_normalized_change']].to_dataframe().reset_index()
    display(summary)
    display(comparison[['model_area_fraction', 'observation_area_fraction']].min('block'))
    summary.to_csv(FIGURE_ROOT / f'{prefix}_summary.csv', index=False)

plot_months = list(comparison_by_month)
max_years = max(comparison_by_month[month].sizes['Y'] for month in plot_months)
month_names = {5: 'May', 11: 'November'}
month_token = '-'.join(f'init{month:02d}' for month in plot_months)
change_plot_specs = [
    (
        'signed_normalized_change', 'signed_normalized_change', 'both',
        r'$(M_2-M_1) / \sigma_{obs,clim}$',
        'Signed lead-year change',
    ),
    (
        'absolute_normalized_change', 'absolute_normalized_change', 'max',
        r'$|M_2-M_1| / \sigma_{obs,clim}$',
        'Absolute lead-year change',
    ),
]

for metric_name, filename_metric, colorbar_extend, colorbar_label, figure_title in change_plot_specs:
    fig, axes = plt.subplots(
        1, len(plot_months),
        figsize=(max(8, 5 * len(plot_months)), max(5, max_years * .25)),
        sharey=True, squeeze=False,
    )
    for col, month in enumerate(plot_months):
        ax = axes[0, col]
        plot_normalized_change(
            comparison_by_month[month], variable=metric_name,
            levels=HEATMAP_COLORBAR_LEVELS[metric_name], ax=ax,
            add_colorbar=False, add_invalid_legend=False,
            title=f"{month_names.get(month, f'Month {month:02d}')} initialization",
        )
        if col:
            ax.set_ylabel('')
            ax.tick_params(labelleft=False)

    mesh = axes[0, 0].images[0]
    bounds = np.asarray(mesh.norm.boundaries)
    fig.subplots_adjust(left=.08, right=.88, bottom=.15, top=.90, wspace=.12)
    colorbar_ax = fig.add_axes([.91, .17, .018, .69])
    colorbar = fig.colorbar(
        mesh, cax=colorbar_ax, ticks=bounds, extend=colorbar_extend,
    )
    colorbar.set_label(colorbar_label)
    colorbar.ax.set_yticklabels([f'{value:g}' for value in bounds])
    invalid_handle = plt.Line2D(
        [], [], marker='s', linestyle='none', markersize=8,
        markerfacecolor=mesh.cmap.get_bad(), markeredgecolor='none',
        label='Invalid / missing',
    )
    fig.legend(handles=[invalid_handle], loc='lower center', frameon=False)
    climatology_label = comparison_by_month[plot_months[0]].attrs.get(
        'normalized_change_climatology', 'available years'
    )
    fig.suptitle(
        f'{field}: {figure_title} normalized by observed annual-mean variability '
        f'({climatology_label})'
    )
    figure_path = FIGURE_ROOT / f'{field}_{month_token}_{period_token}_{filename_metric}.png'
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved normalized-change comparison:', figure_path)


## Compare normalized model and observed lead-year changes

Each point is an initialization year. The 1:1 line denotes equal model and observed
lead-year change; displacement from it is the excess normalized-change diagnostic.


In [ ]:
plot_months = list(comparison_by_month)
case_names = list(dict.fromkeys(
    str(case) for month in plot_months for case in comparison_by_month[month].case.values
))
case_colors = dict(zip(
    case_names, plt.colormaps['tab10'](np.linspace(0, 1, max(2, len(case_names))))
))
default_case_markers = {
    'E3SM-FOSIRL': 'o',
    'E3SM-Reanalysis': 's',
    'E3SM-4DEnVarOcn': 'D',
    'CESM-SMYLE': '^',
    'NMME': 'X',
}
fallback_markers = ('o', 's', 'D', '^', 'v', 'P', 'X', '*')
case_markers = {
    case: default_case_markers.get(case, fallback_markers[index % len(fallback_markers)])
    for index, case in enumerate(case_names)
}
finite_changes = []
for comparison in comparison_by_month.values():
    observed_normalized = (
        comparison.observation_lead_year_change
        / comparison.observation_climatology_std
    )
    for values in (observed_normalized.values, comparison.signed_normalized_change.values):
        values = np.asarray(values, dtype=float)
        finite_changes.extend(values[np.isfinite(values)].tolist())
if not finite_changes:
    raise ValueError('No finite normalized lead-year changes to plot.')
axis_limit = max(1.0, max(abs(value) for value in finite_changes) * 1.05)

fig, axes = plt.subplots(
    1, len(plot_months), figsize=(5 * len(plot_months), 6),
    sharex=True, sharey=True, squeeze=False,
)
month_names = {5: 'May', 11: 'November'}
for col, month in enumerate(plot_months):
    ax = axes[0, col]
    comparison = comparison_by_month[month]
    for case in case_names:
        if case not in set(map(str, comparison.case.values)):
            continue
        data = comparison.sel(case=case)
        observed = np.asarray(
            (data.observation_lead_year_change / data.observation_climatology_std).values,
            dtype=float,
        )
        modeled = np.asarray(data.signed_normalized_change.values, dtype=float)
        valid = np.isfinite(observed) & np.isfinite(modeled)
        ax.scatter(
            observed[valid], modeled[valid], s=46, alpha=.82,
            marker=case_markers[case], color=case_colors[case],
            edgecolor='white', linewidth=.45, label=case, zorder=3,
        )
    ax.plot(
        [-axis_limit, axis_limit], [-axis_limit, axis_limit],
        color='.30', linestyle='--',
        linewidth=1.1, zorder=2,
    )
    ax.axhline(0, color='.75', linewidth=.7, zorder=1)
    ax.axvline(0, color='.75', linewidth=.7, zorder=1)
    ax.set(
        xlim=(-axis_limit, axis_limit), ylim=(-axis_limit, axis_limit),
        xlabel=r'Observed $(O_2-O_1) / \sigma_{obs,clim}$',
        title=f"{month_names.get(month, f'Month {month:02d}')} initialization",
    )
    ax.title.set_fontweight('semibold')
    ax.set_aspect('equal', adjustable='box')
    ax.set_axisbelow(True)
    ax.grid(color='.86', linewidth=.7)
    for spine in ax.spines.values():
        spine.set_color('.35')
        spine.set_linewidth(.8)
axes[0, 0].set_ylabel(r'Model $(M_2-M_1) / \sigma_{obs,clim}$')
fig.suptitle(f'{field}: normalized model versus observed lead-year change', y=.975, fontsize=14)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc='upper center', bbox_to_anchor=(.5, .90),
    ncol=max(1, len(case_names)), fontsize=8.5, frameon=False,
    handletextpad=.5, columnspacing=1.4,
)
fig.subplots_adjust(left=.085, right=.98, bottom=.12, top=.77, wspace=.16)

month_token = '-'.join(f'init{month:02d}' for month in plot_months)
scatter_path = FIGURE_ROOT / f'{field}_{month_token}_{period_token}_normalized_change_scatter.png'
fig.savefig(scatter_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved normalized-change scatter:', scatter_path)


## Evolution of annual global-mean blocks

The black curve is the observed annual-block anomaly. Each colored two-point segment
shows one initialized forecast's first and second annual blocks; color denotes the
initialization year. All anomalies use the observed climatological mean for that
initialization month.


In [ ]:
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

evolution_months = list(comparison_by_month)
evolution_cases = list(dict.fromkeys(
    str(case) for month in evolution_months
    for case in comparison_by_month[month].case.values
))
initialization_years = sorted({
    int(year) for month in evolution_months
    for year in comparison_by_month[month].Y.values
})
if not evolution_months or not evolution_cases or not initialization_years:
    raise ValueError('No annual-block indices are available for the evolution figure.')

year_min, year_max = initialization_years[0], initialization_years[-1]
year_norm = Normalize(vmin=year_min, vmax=year_max if year_max > year_min else year_min + 1)
year_cmap = plt.colormaps['turbo']
month_names = {5: 'May', 11: 'November'}
climatology_years = WORKFLOW_SETTINGS['metric'].get('climatology_years')

fig, axes = plt.subplots(
    len(evolution_cases), len(evolution_months),
    figsize=(7.0 * len(evolution_months), 2.6 * len(evolution_cases)),
    sharex='col', sharey=True, squeeze=False,
)
for col, month in enumerate(evolution_months):
    comparison = comparison_by_month[month]
    block_time = comparison.block_end_time
    if 'case' in block_time.dims:
        block_time = block_time.isel(case=0, drop=True)
    time_x = (
        np.asarray(block_time.dt.year, dtype=float)
        + (np.asarray(block_time.dt.month, dtype=float) - 0.5) / 12.0
    )
    observation = comparison.observation_index
    if 'case' in observation.dims:
        observation = observation.isel(case=0, drop=True)
    climatology = observation
    if climatology_years is not None:
        climatology = climatology.where(
            (climatology.Y >= climatology_years[0])
            & (climatology.Y <= climatology_years[1])
        )
    observed_baseline = climatology.mean(('Y', 'block'), skipna=True)
    if not np.isfinite(float(observed_baseline)):
        raise ValueError(f'No observed climatology is available for init {month:02d}.')
    observed_anomaly = np.asarray(observation - observed_baseline, dtype=float)
    flat_x = time_x.ravel()
    flat_observed = observed_anomaly.ravel()
    finite_observed = np.isfinite(flat_x) & np.isfinite(flat_observed)
    observed_x = np.unique(flat_x[finite_observed])
    observed_y = np.array([
        flat_observed[finite_observed & np.isclose(flat_x, value)].mean()
        for value in observed_x
    ])

    for row, case in enumerate(evolution_cases):
        ax = axes[row, col]
        if case not in set(map(str, comparison.case.values)):
            ax.set_visible(False)
            continue
        modeled = np.asarray(
            comparison.model_index.sel(case=case) - observed_baseline, dtype=float
        )
        for position, initialization_year in enumerate(comparison.Y.values):
            valid = np.isfinite(time_x[position]) & np.isfinite(modeled[position])
            if np.any(valid):
                ax.plot(
                    time_x[position, valid], modeled[position, valid],
                    color=year_cmap(year_norm(int(initialization_year))),
                    linewidth=1.6, marker='o', markersize=2.7, alpha=.88, zorder=2,
                )
        ax.plot(observed_x, observed_y, color='black', linewidth=1.8, zorder=3)
        ax.axhline(0, color='.72', linewidth=.7, zorder=1)
        ax.grid(color='.88', linewidth=.55)
        ax.text(
            .015, .91, case, transform=ax.transAxes, ha='left', va='top',
            fontsize=10.5, fontweight='semibold',
        )
        if row == 0:
            ax.set_title(
                f"{month_names.get(month, f'Month {month:02d}')} initialization",
                fontweight='semibold',
            )
        if col == 0:
            units = comparison.model_index.attrs.get('units', variable.get('units', ''))
            ax.set_ylabel(f'{field} anomaly ({units})' if units else f'{field} anomaly')
        if row == len(evolution_cases) - 1:
            ax.set_xlabel('Verification year')

legend_handles = [
    Line2D([], [], color='black', linewidth=1.8, label='Observed reference'),
    Line2D([], [], color=year_cmap(.65), marker='o', markersize=3,
           linewidth=1.6, label='Initialized two-block forecast'),
]
fig.legend(handles=legend_handles, loc='upper center', ncol=2, frameon=False,
           bbox_to_anchor=(.5, .955))
year_scale = plt.cm.ScalarMappable(norm=year_norm, cmap=year_cmap)
year_scale.set_array([])
colorbar = fig.colorbar(
    year_scale, ax=axes.ravel().tolist(), orientation='horizontal',
    fraction=.025, pad=.075, aspect=45,
)
colorbar.set_label('Initialization year')
fig.suptitle(
    f'{field}: evolution of annual global-mean initialized forecasts',
    y=.995, fontsize=14, fontweight='semibold',
)
fig.subplots_adjust(left=.075, right=.985, bottom=.12, top=.90, hspace=.16, wspace=.10)
evolution_path = (
    FIGURE_ROOT / f'{field}_{month_token}_{period_token}_annual_block_evolution.png'
)
fig.savefig(evolution_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved annual-block evolution:', evolution_path)


## Cleanup

Cached metrics remain available for plotting. Source files are closed inside the
workflow; this idempotent safety cell releases any remaining distributed resources.

See [methodology](../docs/initial_shock_std_index.md) for interpretation and
[the NCL reference](../temp/Compute_Std_Index_Initial_Shock_share.ncl).


In [ ]:
close_notebook_resources(globals())
print('Closed notebook Dask resources.')
